<a href="https://colab.research.google.com/github/Joyce-VFS/Portfolio/blob/Academic/Option2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding

import numpy as np
import string

# ---------------------------------------------------
# 1. LOAD AND PREPARE TEXT
# ---------------------------------------------------

# This is the text the model will train on.
# Load the Shakespeare dataset downloaded with wget
with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Convert the entire text to lowercase.
# This makes the dataset more consistent because 'A' and 'a' become the same character.
# Reducing variation in the data makes the model easier to train.
text = text.lower()

# Remove any non-printable or unusual characters.
# string.printable contains letters, digits, punctuation, and common whitespace.
# Keeping only printable characters ensures the model doesn't have to learn
# from rare or unexpected symbols (like emojis or hidden control characters),
# which can distort the vocabulary and hurt training performance.
text = ''.join(c for c in text if c in string.printable)

# Extract the set of all unique characters in the cleaned text.
# Using set() removes duplicates and sorted() puts them in a consistent order.
# This "character vocabulary" defines all possible characters the model can predict.
chars = sorted(list(set(text)))
vocab_size = len(chars)  # total number of unique characters

# Create two lookup tables:
# 1) char_to_idx maps each character → a unique integer ID.
# 2) idx_to_char maps each integer ID → the corresponding character.
# Neural networks cannot work directly with letters, so we convert text to numbers before training,
# and convert numbers back to characters during text generation.
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

# Convert the entire text into its numerical representation.
# For example: "hello" → [7, 4, 11, 11, 14] depending on the vocabulary mapping.
# The result is a NumPy array that the model can train on.
encoded_text = np.array([char_to_idx[c] for c in text])

# ---------------------------------------------------
# 2. CREATE TRAINING SEQUENCES
# ---------------------------------------------------

# The model does not read the entire text at once.
# Instead, it reads fixed-size "windows" of characters and tries to predict the next character.
sequence_length = 40  # the length of each input sequence
sequences = []         # will store all input sequences
labels = []            # will store the next character after each sequence

# Slide a window of length 40 across the text.
# For each window (sequence), the label is the character that comes right after the window.
# Example: if the window contains "to be or not to be, that is th"
# then the label might be "e".
for i in range(len(encoded_text) - sequence_length):
    seq = encoded_text[i : i + sequence_length]      # 40-character sequence
    next_char = encoded_text[i + sequence_length]    # the character to predict

    sequences.append(seq)
    labels.append(next_char)

# Convert the list of sequences into a 2D NumPy array.
X = np.array(sequences)

# Convert the labels into one-hot encoded format.
# One-hot encoding represents each character as a vector where one position is 1
# (the correct character) and all others are 0.
# This matches the softmax output layer, which predicts a probability for each character.
y = tf.keras.utils.to_categorical(labels, num_classes=vocab_size)

# ---------------------------------------------------
# 3. BUILD LSTM MODEL
# ---------------------------------------------------

# Each character ID will be embedded into a vector of size 64.
# The LSTM layers will each have 128 hidden units.
embedding_dim = 64
lstm_units = 128

# Build the model using the Sequential API.
model = Sequential([
    # Embedding layer turns integer-encoded characters into dense vectors.
    # This helps the network learn relationships between characters.
    Embedding(vocab_size, embedding_dim, input_length=sequence_length),

    # First LSTM layer reads the sequence and returns another sequence
    # because we are stacking a second LSTM on top of it.
    LSTM(lstm_units, return_sequences=True),

    # Second LSTM layer processes the output of the first LSTM and returns
    # only the final hidden state, which summarizes the information in the sequence.
    LSTM(lstm_units),

    # Final Dense layer predicts the next character.
    # Softmax converts output into a probability distribution over all characters.
    Dense(vocab_size, activation='softmax')
])

# Compile the model using the Adam optimizer and categorical crossentropy loss.
# This combination is standard for sequence prediction problems.
model.compile(optimizer='adam',
              loss='categorical_crossentropy')

# Print a summary to see the layer structure and number of trainable parameters.
print(model.summary())

# ---------------------------------------------------
# 4. TRAIN THE MODEL
# ---------------------------------------------------

# Train the model on the input sequences (X) and one-hot labels (y).
# A batch size of 64 is common, and 20 epochs is enough for a small dataset.
model.fit(X, y, batch_size=64, epochs=20)

# ---------------------------------------------------
# 5. TEXT GENERATION FUNCTION
# ---------------------------------------------------

def generate_text(seed, length=200, temperature=1.0):
    """
    Generates text character-by-character using the trained LSTM model.
    'seed' is the starting text.
    'length' is how many new characters to generate.
    'temperature' controls randomness (lower = less random, higher = more random).
    """
    # Convert the seed text to lowercase for consistency with training data.
    result = seed.lower()

    # Convert each seed character to its integer ID.
    encoded = [char_to_idx[c] for c in seed.lower()]

    for _ in range(length):
        # The model expects sequences of exactly 40 characters.
        # If the seed is shorter, we pad it with zeros at the front.
        sample_window = encoded[-sequence_length:]
        if len(sample_window) < sequence_length:
            pad_len = sequence_length - len(sample_window)
            sample_window = [0] * pad_len + sample_window

        # Convert to NumPy array with shape (1, sequence_length),
        # since the model expects a batch dimension.
        sample_window = np.array(sample_window).reshape(1, -1)

        # Predict the probability distribution for the next character.
        preds = model.predict(sample_window, verbose=0)[0]

        # Apply temperature to control randomness.
        # Higher temperature makes the distribution flatter,
        # causing the model to take more creative/less certain guesses.
        preds = np.log(preds + 1e-8) / temperature
        exp_preds = np.exp(preds)
        preds = exp_preds / np.sum(exp_preds)

        # Randomly choose a character index according to the probability distribution.
        next_idx = np.random.choice(range(vocab_size), p=preds)
        next_char = idx_to_char[next_idx]

        # Append the new character to both the numeric sequence and the output string.
        encoded.append(next_idx)
        result += next_char

    return result

# ---------------------------------------------------
# 6. TEST THE MODEL BY GENERATING TEXT
# ---------------------------------------------------

# Provide a starting sentence ("seed") and generate new text from it.
seed_text = "to be or not to be"
generated = generate_text(seed_text, length=300, temperature=0.8)
print(generated)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/20
17428/17428 ━━━━━━━━━━━━━━━━━━━━ 2557s 147ms/step - loss: 2.0367
Epoch 2/20
  522/17428 ━━━━━━━━━━━━━━━━━━━━ 41:34 148ms/step - loss: 1.4962

In [5]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O shakespeare.txt


--2025-11-17 15:19:47--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘shakespeare.txt’

shakespeare.txt     100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2025-11-17 15:19:47 (35.8 MB/s) - ‘shakespeare.txt’ saved [1115394/1115394]



In [7]:
# list files in current folder and show file size
!ls -lh shakespeare.txt


-rw-r--r-- 1 root root 1.1M Nov 17 15:19 shakespeare.txt


Presentation:

* Show generated text samples (3-5 examples)
* Explain: How does LSTM ”remember” the style?
* Compare outputs with different temperatures
* What patterns did the model learn?

In [8]:
# read the file into a Python string
with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

# quick checks
print("Characters in dataset:", len(text))
print("First 500 characters:\n")
print(text[:500])


Characters in dataset: 1115394
First 500 characters:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor
